# C2 Statistical Validation and PP1 Evidence

This notebook statistically validates the current **Component 2 prototype selector comparisons** and prepares concise PP1-ready evidence.

## Scope

The notebook performs:

- paired Wilcoxon signed-rank tests across products,
- paired bootstrap 95% confidence intervals,
- Holm correction for multiple comparisons,
- targeted ablation validation,
- minority reserved-slot sensitivity,
- concise PP1 evidence-table generation.

## Important Interpretation Limitation

The current C1 aspect/sentiment information is synthetic integration-test data.

Therefore, statistical results involving:

- aspect coverage,
- supported-minority recall,
- priority-minority coverage,
- opinion-distribution fidelity,

are **prototype diagnostic evidence only** and must not be presented as final research findings.

The smartphone pilot contains 10 products. Statistical tests are therefore exploratory and should be interpreted together with effect direction, magnitude and confidence intervals rather than p-values alone.


In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

from scipy.stats import wilcoxon
from scipy.spatial.distance import jensenshannon
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_columns", 50)

print("Statistical validation environment ready")


Statistical validation environment ready


## 1. Resolve Paths and Load Evaluation Artifacts


In [2]:
CURRENT_DIR = Path.cwd()

if (CURRENT_DIR / "component_2").exists():
    COMPONENT_ROOT = CURRENT_DIR / "component_2"

elif CURRENT_DIR.name == "component_2":
    COMPONENT_ROOT = CURRENT_DIR

elif (
    CURRENT_DIR.name == "notebooks"
    and CURRENT_DIR.parent.name == "component_2"
):
    COMPONENT_ROOT = CURRENT_DIR.parent

else:
    raise RuntimeError(
        f"Could not locate component_2 from: {CURRENT_DIR}"
    )

SELECTOR_EVAL_DIR = (
    COMPONENT_ROOT
    / "results"
    / "selector_evaluation"
)

SELECTOR_DIR = (
    COMPONENT_ROOT
    / "results"
    / "selector"
)

INTEGRATION_DIR = (
    COMPONENT_ROOT
    / "results"
    / "integration"
)

STAT_DIR = (
    COMPONENT_ROOT
    / "results"
    / "statistical_validation"
)

STAT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

BASELINE_PRODUCT_PATH = (
    SELECTOR_EVAL_DIR
    / "baseline_comparison_by_product.csv"
)

ABLATION_PRODUCT_PATH = (
    SELECTOR_EVAL_DIR
    / "ablation_by_product.csv"
)

SELECTOR_READY_PATH = (
    SELECTOR_DIR
    / "selector_ready_reviews.csv"
)

EMBEDDING_PATH = (
    INTEGRATION_DIR
    / "smartphone_review_embeddings.npy"
)

C1_MOCK_PATH = (
    INTEGRATION_DIR
    / "c1_mock_aspect_sentiment.csv"
)

SUPPORTED_MINORITY_PATH = (
    INTEGRATION_DIR
    / "mock_supported_minority_groups.csv"
)

required_paths = {
    "baseline_by_product": BASELINE_PRODUCT_PATH,
    "ablation_by_product": ABLATION_PRODUCT_PATH,
    "selector_ready": SELECTOR_READY_PATH,
    "embeddings": EMBEDDING_PATH,
    "mock_c1": C1_MOCK_PATH,
    "supported_minority": SUPPORTED_MINORITY_PATH,
}

print("Current directory:", CURRENT_DIR)
print("Component root:", COMPONENT_ROOT)

for name, path in required_paths.items():
    print(f"{name}: {path.exists()} -> {path}")

missing = [
    str(path)
    for path in required_paths.values()
    if not path.exists()
]

if missing:
    raise FileNotFoundError(
        "Missing required artifacts:\n"
        + "\n".join(missing)
    )

print("\nAll required statistical-validation artifacts found.")


Current directory: d:\Research\Repository\J26-DS-349\component_2\notebooks
Component root: d:\Research\Repository\J26-DS-349\component_2
baseline_by_product: True -> d:\Research\Repository\J26-DS-349\component_2\results\selector_evaluation\baseline_comparison_by_product.csv
ablation_by_product: True -> d:\Research\Repository\J26-DS-349\component_2\results\selector_evaluation\ablation_by_product.csv
selector_ready: True -> d:\Research\Repository\J26-DS-349\component_2\results\selector\selector_ready_reviews.csv
embeddings: True -> d:\Research\Repository\J26-DS-349\component_2\results\integration\smartphone_review_embeddings.npy
mock_c1: True -> d:\Research\Repository\J26-DS-349\component_2\results\integration\c1_mock_aspect_sentiment.csv
supported_minority: True -> d:\Research\Repository\J26-DS-349\component_2\results\integration\mock_supported_minority_groups.csv

All required statistical-validation artifacts found.


In [3]:
baseline_product_df = pd.read_csv(
    BASELINE_PRODUCT_PATH
)

ablation_product_df = pd.read_csv(
    ABLATION_PRODUCT_PATH
)

selector_ready_df = pd.read_csv(
    SELECTOR_READY_PATH
)

embedding_matrix = np.load(
    EMBEDDING_PATH
)

c1_mock_df = pd.read_csv(
    C1_MOCK_PATH
)

supported_minority_df = pd.read_csv(
    SUPPORTED_MINORITY_PATH
)

print(
    "Baseline product rows:",
    len(baseline_product_df)
)

print(
    "Ablation product rows:",
    len(ablation_product_df)
)

print(
    "Selector-ready rows:",
    len(selector_ready_df)
)

print(
    "Embedding shape:",
    embedding_matrix.shape
)

print(
    "Products:",
    selector_ready_df["parent_asin"].nunique()
)


Baseline product rows: 60
Ablation product rows: 50
Selector-ready rows: 896
Embedding shape: (896, 384)
Products: 10


In [4]:
EXPECTED_MAIN_SELECTORS = {
    "ObservedHelpfulTopK",
    "PredictedUsefulnessTopK",
    "HelpfulnessMMR",
    "AspectCoverage",
    "ProposedNoMinority",
    "FullProposed",
}

EXPECTED_ABLATIONS = {
    "FullProposed",
    "NoMinority",
    "NoRedundancy",
    "NoCoverage",
    "NoRepresentativeness",
}

assert EXPECTED_MAIN_SELECTORS.issubset(
    set(baseline_product_df["selector"])
)

assert EXPECTED_ABLATIONS.issubset(
    set(ablation_product_df["selector"])
)

assert baseline_product_df["parent_asin"].nunique() == 10
assert ablation_product_df["parent_asin"].nunique() == 10

for selector_name in EXPECTED_MAIN_SELECTORS:
    assert (
        baseline_product_df[
            baseline_product_df["selector"] == selector_name
        ]["parent_asin"].nunique()
        == 10
    )

for selector_name in EXPECTED_ABLATIONS:
    assert (
        ablation_product_df[
            ablation_product_df["selector"] == selector_name
        ]["parent_asin"].nunique()
        == 10
    )

print(
    "Statistical-validation input checks passed."
)


Statistical-validation input checks passed.


## 2. Statistical Helper Functions

For each paired product-level comparison:

- the Wilcoxon signed-rank test evaluates whether paired values differ,
- paired bootstrap resampling estimates a 95% confidence interval for the mean **benefit-oriented difference**,
- Holm correction controls the family-wise error rate across each experiment family.

For higher-is-better metrics, benefit difference is:

`Full Proposed − Comparator`

For lower-is-better metrics, benefit difference is:

`Comparator − Full Proposed`

Therefore, **positive benefit difference always favors Full Proposed**.


In [5]:
METRIC_DIRECTIONS = {
    "mean_usefulness_norm": 1,
    "mean_representativeness_norm": 1,
    "mean_observed_log_helpful": 1,
    "aspect_coverage": 1,
    "supported_minority_recall": 1,
    "priority_minority_coverage": 1,
    "mean_pairwise_similarity": -1,
    "opinion_js_divergence": -1,
}

METRIC_LABELS = {
    "mean_usefulness_norm": "Predicted usefulness",
    "mean_representativeness_norm": "Semantic representativeness",
    "mean_observed_log_helpful": "Observed log helpfulness",
    "aspect_coverage": "Aspect coverage",
    "supported_minority_recall": "Supported-minority recall",
    "priority_minority_coverage": "Priority-minority coverage",
    "mean_pairwise_similarity": "Pairwise similarity",
    "opinion_js_divergence": "Opinion JSD",
}

MAIN_METRICS = list(
    METRIC_DIRECTIONS.keys()
)

print(
    "Metrics configured:",
    len(MAIN_METRICS)
)


Metrics configured: 8


In [6]:
def paired_vectors(
    df,
    selector_a,
    selector_b,
    metric,
):
    a = (
        df[
            df["selector"] == selector_a
        ][
            ["parent_asin", metric]
        ]
        .rename(
            columns={
                metric: "a"
            }
        )
    )

    b = (
        df[
            df["selector"] == selector_b
        ][
            ["parent_asin", metric]
        ]
        .rename(
            columns={
                metric: "b"
            }
        )
    )

    paired = (
        a.merge(
            b,
            on="parent_asin",
            how="inner",
            validate="one_to_one",
        )
        .dropna(
            subset=["a", "b"]
        )
        .sort_values("parent_asin")
        .reset_index(drop=True)
    )

    return paired


In [7]:
def safe_wilcoxon(
    a_values,
    b_values,
):
    a = np.asarray(
        a_values,
        dtype=float,
    )

    b = np.asarray(
        b_values,
        dtype=float,
    )

    diffs = a - b

    if len(diffs) == 0:
        return np.nan, np.nan

    if np.allclose(
        diffs,
        0.0,
    ):
        return 0.0, 1.0

    result = wilcoxon(
        a,
        b,
        alternative="two-sided",
        zero_method="wilcox",
        method="auto",
    )

    return (
        float(result.statistic),
        float(result.pvalue),
    )


In [8]:
def paired_bootstrap_ci(
    a_values,
    b_values,
    direction,
    n_boot=10000,
    seed=349,
):
    a = np.asarray(
        a_values,
        dtype=float,
    )

    b = np.asarray(
        b_values,
        dtype=float,
    )

    benefit_diff = (
        direction
        * (a - b)
    )

    if len(benefit_diff) == 0:
        return (
            np.nan,
            np.nan,
            np.nan,
        )

    rng = np.random.default_rng(
        seed
    )

    samples = np.empty(
        n_boot,
        dtype=float,
    )

    n = len(
        benefit_diff
    )

    for i in range(
        n_boot
    ):
        indices = rng.integers(
            0,
            n,
            size=n,
        )

        samples[i] = float(
            np.mean(
                benefit_diff[
                    indices
                ]
            )
        )

    return (
        float(
            np.mean(
                benefit_diff
            )
        ),
        float(
            np.quantile(
                samples,
                0.025,
            )
        ),
        float(
            np.quantile(
                samples,
                0.975,
            )
        ),
    )


In [9]:
def holm_adjust(
    p_values
):
    p = np.asarray(
        p_values,
        dtype=float,
    )

    adjusted = np.full(
        len(p),
        np.nan,
        dtype=float,
    )

    valid_mask = np.isfinite(
        p
    )

    valid_indices = np.where(
        valid_mask
    )[0]

    if len(valid_indices) == 0:
        return adjusted

    valid_p = p[
        valid_indices
    ]

    order = np.argsort(
        valid_p
    )

    sorted_p = valid_p[
        order
    ]

    m = len(
        sorted_p
    )

    raw_adjusted = np.array(
        [
            (m - i)
            * sorted_p[i]
            for i in range(m)
        ],
        dtype=float,
    )

    monotonic_adjusted = np.maximum.accumulate(
        raw_adjusted
    )

    monotonic_adjusted = np.minimum(
        monotonic_adjusted,
        1.0,
    )

    back = np.empty(
        m,
        dtype=float,
    )

    back[
        order
    ] = monotonic_adjusted

    adjusted[
        valid_indices
    ] = back

    return adjusted


In [10]:
def run_paired_comparison(
    df,
    selector_a,
    selector_b,
    metric,
    bootstrap_seed=349,
):
    paired = paired_vectors(
        df=df,
        selector_a=selector_a,
        selector_b=selector_b,
        metric=metric,
    )

    direction = METRIC_DIRECTIONS[
        metric
    ]

    statistic, p_value = safe_wilcoxon(
        paired["a"].values,
        paired["b"].values,
    )

    (
        mean_benefit_diff,
        ci_low,
        ci_high,
    ) = paired_bootstrap_ci(
        paired["a"].values,
        paired["b"].values,
        direction=direction,
        n_boot=10000,
        seed=bootstrap_seed,
    )

    raw_diff = (
        paired["a"].values
        - paired["b"].values
    )

    benefit_diff = (
        direction
        * raw_diff
    )

    return {
        "selector_a": selector_a,
        "selector_b": selector_b,
        "metric": metric,
        "metric_label": METRIC_LABELS[metric],
        "n_products": int(len(paired)),
        "mean_a": float(paired["a"].mean()),
        "mean_b": float(paired["b"].mean()),
        "mean_raw_difference_a_minus_b": float(
            np.mean(raw_diff)
        ),
        "mean_benefit_difference": float(
            mean_benefit_diff
        ),
        "benefit_ci_95_low": float(ci_low),
        "benefit_ci_95_high": float(ci_high),
        "wilcoxon_statistic": float(statistic),
        "wilcoxon_p": float(p_value),
        "a_better_products": int(
            np.sum(
                benefit_diff > 1e-12
            )
        ),
        "ties": int(
            np.sum(
                np.isclose(
                    benefit_diff,
                    0.0,
                    atol=1e-12,
                )
            )
        ),
        "a_worse_products": int(
            np.sum(
                benefit_diff < -1e-12
            )
        ),
    }


## 3. Full Proposed vs Main Baselines

The Full Proposed selector is compared with:

- Observed Helpful-Vote Top-K,
- Predicted-Usefulness Top-K,
- Helpfulness + MMR,
- Aspect-Coverage,
- Proposed without Minority.

These tests are exploratory because the pilot includes only 10 products.


In [11]:
MAIN_COMPARATORS = [
    "ObservedHelpfulTopK",
    "PredictedUsefulnessTopK",
    "HelpfulnessMMR",
    "AspectCoverage",
    "ProposedNoMinority",
]

main_stat_records = []

for comparator in MAIN_COMPARATORS:

    for metric_index, metric in enumerate(
        MAIN_METRICS
    ):
        main_stat_records.append(
            run_paired_comparison(
                df=baseline_product_df,
                selector_a="FullProposed",
                selector_b=comparator,
                metric=metric,
                bootstrap_seed=(
                    349
                    + metric_index
                    + 100 * MAIN_COMPARATORS.index(comparator)
                ),
            )
        )

main_stat_tests_df = pd.DataFrame(
    main_stat_records
)

main_stat_tests_df[
    "holm_p"
] = holm_adjust(
    main_stat_tests_df[
        "wilcoxon_p"
    ].values
)

main_stat_tests_df[
    "holm_significant_0_05"
] = (
    main_stat_tests_df[
        "holm_p"
    ] < 0.05
)

display(
    main_stat_tests_df.head(20)
)

print(
    "Main statistical tests:",
    len(main_stat_tests_df)
)


,selector_a,selector_b,metric,metric_label,n_products,mean_a,mean_b,mean_raw_difference_a_minus_b,mean_benefit_difference,benefit_ci_95_low,benefit_ci_95_high,wilcoxon_statistic,wilcoxon_p,a_better_products,ties,a_worse_products,holm_p,holm_significant_0_05
0,FullProposed,ObservedHelpfulTopK,mean_usefulness_norm,Predicted usefulness,10,0.742725,0.768678,-0.025953,-0.025953,-0.075031,0.020064,21.0,0.556641,5,0,5,1.000000,False
1,FullProposed,ObservedHelpfulTopK,mean_representativeness_norm,Semantic representativeness,10,0.766597,0.681043,0.085554,0.085554,0.037597,0.138259,4.0,0.013672,9,0,1,0.341797,False
2,FullProposed,ObservedHelpfulTopK,mean_observed_log_helpful,Observed log helpfulness,10,1.824493,2.787884,-0.963391,-0.963391,-1.164607,-0.771662,0.0,0.001953,0,0,10,0.078125,False
3,FullProposed,ObservedHelpfulTopK,aspect_coverage,Aspect coverage,10,0.933333,0.866667,0.066667,0.066667,-0.016667,0.150000,4.0,0.500000,4,5,1,1.000000,False
4,FullProposed,ObservedHelpfulTopK,supported_minority_recall,Supported-minority recall,10,0.416744,0.322098,0.094646,0.094646,0.016653,0.180000,4.0,0.109375,6,3,1,1.000000,False
5,FullProposed,ObservedHelpfulTopK,priority_minority_coverage,Priority-minority coverage,10,1.000000,0.200000,0.800000,0.800000,0.650000,0.950000,0.0,0.001953,10,0,0,0.078125,False
6,FullProposed,ObservedHelpfulTopK,mean_pairwise_similarity,Pairwise similarity,10,0.492611,0.411797,0.080814,-0.080814,-0.135977,-0.029343,6.0,0.027344,2,0,8,0.601562,False
7,FullProposed,ObservedHelpfulTopK,opinion_js_divergence,Opinion JSD,10,0.336771,0.314594,0.022176,-0.022176,-0.057353,0.018792,17.0,0.322266,3,0,7,1.000000,False
8,FullProposed,PredictedUsefulnessTopK,mean_usefulness_norm,Predicted usefulness,10,0.742725,0.859078,-0.116353,-0.116353,-0.152127,-0.079277,0.0,0.001953,0,0,10,0.078125,False
9,FullProposed,PredictedUsefulnessTopK,mean_representativeness_norm,Semantic representativeness,10,0.766597,0.711964,0.054633,0.054633,0.017112,0.096114,7.0,0.037109,8,0,2,0.667969,False


Main statistical tests: 40


### Main Comparison Interpretation

A positive `mean_benefit_difference` favors the Full Proposed selector after metric direction is taken into account.

Because Holm correction is conservative and the pilot contains only 10 products, non-significant p-values do **not** imply that the observed prototype differences are meaningless. The paired effect direction and bootstrap interval should also be inspected.


In [12]:
main_key_view = (
    main_stat_tests_df[
        [
            "selector_b",
            "metric_label",
            "mean_a",
            "mean_b",
            "mean_benefit_difference",
            "benefit_ci_95_low",
            "benefit_ci_95_high",
            "wilcoxon_p",
            "holm_p",
            "a_better_products",
            "ties",
            "a_worse_products",
        ]
    ]
    .copy()
)

display(
    main_key_view
)


,selector_b,metric_label,mean_a,mean_b,mean_benefit_difference,benefit_ci_95_low,benefit_ci_95_high,wilcoxon_p,holm_p,a_better_products,ties,a_worse_products
0,ObservedHelpfulTopK,Predicted usefulness,0.742725,0.768678,-0.025953,-0.075031,0.020064,0.556641,1.000000,5,0,5
1,ObservedHelpfulTopK,Semantic representativeness,0.766597,0.681043,0.085554,0.037597,0.138259,0.013672,0.341797,9,0,1
2,ObservedHelpfulTopK,Observed log helpfulness,1.824493,2.787884,-0.963391,-1.164607,-0.771662,0.001953,0.078125,0,0,10
3,ObservedHelpfulTopK,Aspect coverage,0.933333,0.866667,0.066667,-0.016667,0.150000,0.500000,1.000000,4,5,1
4,ObservedHelpfulTopK,Supported-minority recall,0.416744,0.322098,0.094646,0.016653,0.180000,0.109375,1.000000,6,3,1
5,ObservedHelpfulTopK,Priority-minority coverage,1.000000,0.200000,0.800000,0.650000,0.950000,0.001953,0.078125,10,0,0
6,ObservedHelpfulTopK,Pairwise similarity,0.492611,0.411797,-0.080814,-0.135977,-0.029343,0.027344,0.601562,2,0,8
7,ObservedHelpfulTopK,Opinion JSD,0.336771,0.314594,-0.022176,-0.057353,0.018792,0.322266,1.000000,3,0,7
8,PredictedUsefulnessTopK,Predicted usefulness,0.742725,0.859078,-0.116353,-0.152127,-0.079277,0.001953,0.078125,0,0,10
9,PredictedUsefulnessTopK,Semantic representativeness,0.766597,0.711964,0.054633,0.017112,0.096114,0.037109,0.667969,8,0,2


## 4. Targeted Ablation Validation

Each ablation is evaluated on the metric that directly corresponds to the removed mechanism.

- No Minority → supported-minority recall and priority-minority coverage
- No Coverage → aspect coverage
- No Redundancy → pairwise similarity
- No Representativeness → semantic representativeness

This targeted design avoids inflating the number of unnecessary statistical tests.


In [13]:
ABLATION_TARGETS = [
    (
        "NoMinority",
        "supported_minority_recall",
    ),
    (
        "NoMinority",
        "priority_minority_coverage",
    ),
    (
        "NoCoverage",
        "aspect_coverage",
    ),
    (
        "NoRedundancy",
        "mean_pairwise_similarity",
    ),
    (
        "NoRepresentativeness",
        "mean_representativeness_norm",
    ),
]

ablation_stat_records = []

for i, (
    comparator,
    metric,
) in enumerate(
    ABLATION_TARGETS
):
    ablation_stat_records.append(
        run_paired_comparison(
            df=ablation_product_df,
            selector_a="FullProposed",
            selector_b=comparator,
            metric=metric,
            bootstrap_seed=700 + i,
        )
    )

ablation_stat_tests_df = pd.DataFrame(
    ablation_stat_records
)

ablation_stat_tests_df[
    "holm_p"
] = holm_adjust(
    ablation_stat_tests_df[
        "wilcoxon_p"
    ].values
)

ablation_stat_tests_df[
    "holm_significant_0_05"
] = (
    ablation_stat_tests_df[
        "holm_p"
    ] < 0.05
)

display(
    ablation_stat_tests_df
)


,selector_a,selector_b,metric,metric_label,n_products,mean_a,mean_b,mean_raw_difference_a_minus_b,mean_benefit_difference,benefit_ci_95_low,benefit_ci_95_high,wilcoxon_statistic,wilcoxon_p,a_better_products,ties,a_worse_products,holm_p,holm_significant_0_05
0,FullProposed,NoMinority,supported_minority_recall,Supported-minority recall,10,0.416744,0.281072,0.135672,0.135672,0.082168,0.195125,0.0,0.003906,9,1,0,0.011719,True
1,FullProposed,NoMinority,priority_minority_coverage,Priority-minority coverage,10,1.000000,0.200000,0.800000,0.800000,0.650000,0.950000,0.0,0.001953,10,0,0,0.009766,True
2,FullProposed,NoCoverage,aspect_coverage,Aspect coverage,10,0.933333,0.833333,0.100000,0.100000,0.016667,0.183333,0.0,0.125000,4,6,0,0.218750,False
3,FullProposed,NoRedundancy,mean_pairwise_similarity,Pairwise similarity,10,0.492611,0.514537,-0.021926,0.021926,0.000693,0.051679,4.0,0.109375,6,3,1,0.218750,False
4,FullProposed,NoRepresentativeness,mean_representativeness_norm,Semantic representativeness,10,0.766597,0.647577,0.119019,0.119019,0.081538,0.154691,0.0,0.001953,10,0,0,0.009766,True


## 5. Minority Reserved-Slot Sensitivity

Notebook 06 showed that changing only the minority score weight did not change the selected set under the current prototype configuration.

To test the **reservation mechanism itself**, this section keeps all Full Proposed weights fixed and varies only the number of reserved minority slots:

- 0
- 1
- 2
- 3

This isolates the effect of explicit reservation more clearly than the previous minority-weight sensitivity.


In [14]:
def parse_pipe_set(
    value
):
    if pd.isna(value):
        return set()

    return {
        item
        for item in str(value).split("|")
        if item
    }


supported_minority_df[
    "aspect_sentiment_key"
] = (
    supported_minority_df[
        "aspect_category"
    ].astype(str)
    + "::"
    + supported_minority_df[
        "sentiment"
    ].astype(str)
)

c1_mock_df[
    "aspect_sentiment_key"
] = (
    c1_mock_df[
        "aspect_category"
    ].astype(str)
    + "::"
    + c1_mock_df[
        "sentiment"
    ].astype(str)
)

supported_keys_by_product = (
    supported_minority_df
    .groupby("parent_asin")[
        "aspect_sentiment_key"
    ]
    .apply(set)
    .to_dict()
)


In [15]:
def build_priority_minority_map(
    num_priority_groups=2,
):
    priority_map = {}

    for product_id, group in (
        supported_minority_df
        .groupby("parent_asin")
    ):
        sorted_group = (
            group
            .sort_values(
                [
                    "prevalence",
                    "support_count",
                    "avg_confidence",
                ],
                ascending=[
                    True,
                    False,
                    False,
                ],
            )
            .drop_duplicates(
                subset=[
                    "aspect_sentiment_key"
                ]
            )
        )

        priority_map[
            product_id
        ] = set(
            sorted_group[
                "aspect_sentiment_key"
            ]
            .head(
                num_priority_groups
            )
            .tolist()
        )

    return priority_map


priority_minority_by_product = (
    build_priority_minority_map()
)

print(
    "Priority maps:",
    len(priority_minority_by_product)
)


Priority maps: 10


In [16]:
def candidate_redundancy(
    candidate_embedding_row,
    selected_embedding_rows,
):
    if len(
        selected_embedding_rows
    ) == 0:
        return 0.0

    candidate_vector = (
        embedding_matrix[
            int(candidate_embedding_row)
        ]
        .reshape(1, -1)
    )

    selected_vectors = (
        embedding_matrix[
            np.asarray(
                selected_embedding_rows,
                dtype=int,
            )
        ]
    )

    similarities = cosine_similarity(
        candidate_vector,
        selected_vectors,
    )[0]

    return max(
        0.0,
        float(np.max(similarities)),
    )


def mean_pairwise_similarity(
    embedding_rows
):
    embedding_rows = list(
        embedding_rows
    )

    if len(
        embedding_rows
    ) < 2:
        return 0.0

    vectors = embedding_matrix[
        np.asarray(
            embedding_rows,
            dtype=int,
        )
    ]

    similarity_matrix = cosine_similarity(
        vectors
    )

    upper = np.triu_indices(
        len(embedding_rows),
        k=1,
    )

    return float(
        np.mean(
            similarity_matrix[
                upper
            ]
        )
    )


In [17]:
def product_opinion_jsd(
    product_id,
    selected_review_ids,
):
    product_review_ids = set(
        selector_ready_df[
            selector_ready_df[
                "parent_asin"
            ] == product_id
        ]["review_id"]
    )

    full_c1 = c1_mock_df[
        c1_mock_df[
            "review_id"
        ].isin(
            product_review_ids
        )
    ]

    selected_c1 = c1_mock_df[
        c1_mock_df[
            "review_id"
        ].isin(
            set(selected_review_ids)
        )
    ]

    categories = sorted(
        set(
            full_c1[
                "aspect_sentiment_key"
            ]
        )
        |
        set(
            selected_c1[
                "aspect_sentiment_key"
            ]
        )
    )

    if not categories:
        return np.nan

    full_counts = (
        full_c1[
            "aspect_sentiment_key"
        ]
        .value_counts()
    )

    selected_counts = (
        selected_c1[
            "aspect_sentiment_key"
        ]
        .value_counts()
    )

    p = np.array(
        [
            full_counts.get(
                key,
                0,
            )
            for key in categories
        ],
        dtype=float,
    )

    q = np.array(
        [
            selected_counts.get(
                key,
                0,
            )
            for key in categories
        ],
        dtype=float,
    )

    if (
        p.sum() <= 0
        or q.sum() <= 0
    ):
        return np.nan

    p = p / p.sum()
    q = q / q.sum()

    return float(
        jensenshannon(
            p,
            q,
            base=2.0,
        ) ** 2
    )


In [18]:
FULL_WEIGHTS = {
    "usefulness": 0.30,
    "representativeness": 0.25,
    "coverage": 0.20,
    "minority": 0.25,
    "redundancy": 0.20,
}

SELECT_K = 5
RESERVED_SLOT_VALUES = [
    0,
    1,
    2,
    3,
]


def candidate_score(
    row,
    selected_embedding_rows,
    covered_aspects,
    total_product_aspects,
    covered_priority_groups,
    priority_groups,
):
    candidate_aspects = parse_pipe_set(
        row[
            "aspect_categories"
        ]
    )

    candidate_minority_groups = parse_pipe_set(
        row[
            "supported_minority_groups"
        ]
    )

    new_aspects = (
        candidate_aspects
        - covered_aspects
    )

    uncovered_priority = (
        priority_groups
        - covered_priority_groups
    )

    new_priority_groups = (
        candidate_minority_groups
        & uncovered_priority
    )

    coverage_gain = (
        len(new_aspects)
        /
        max(
            len(total_product_aspects),
            1,
        )
    )

    minority_gain = (
        len(new_priority_groups)
        /
        len(priority_groups)
        if priority_groups
        else 0.0
    )

    redundancy = candidate_redundancy(
        row["embedding_row"],
        selected_embedding_rows,
    )

    score = (
        FULL_WEIGHTS["usefulness"]
        * float(
            row["usefulness_norm"]
        )
        +
        FULL_WEIGHTS["representativeness"]
        * float(
            row["representativeness_norm"]
        )
        +
        FULL_WEIGHTS["coverage"]
        * coverage_gain
        +
        FULL_WEIGHTS["minority"]
        * minority_gain
        -
        FULL_WEIGHTS["redundancy"]
        * redundancy
    )

    return {
        "selection_score": float(score),
        "coverage_gain": float(coverage_gain),
        "minority_gain": float(minority_gain),
        "redundancy": float(redundancy),
    }


In [19]:
def select_with_reserved_slots(
    product_df,
    budget,
    reserved_slots,
):
    product_df = (
        product_df
        .copy()
        .reset_index(drop=True)
    )

    product_id = product_df.loc[
        0,
        "parent_asin",
    ]

    priority_groups = (
        priority_minority_by_product
        .get(
            product_id,
            set(),
        )
    )

    total_product_aspects = set()

    for value in product_df[
        "aspect_categories"
    ]:
        total_product_aspects |= (
            parse_pipe_set(value)
        )

    selected_indices = []
    selected_embedding_rows = []
    covered_aspects = set()
    covered_priority_groups = set()

    def evaluate_candidate(
        idx
    ):
        row = product_df.loc[idx]

        diagnostics = candidate_score(
            row=row,
            selected_embedding_rows=
                selected_embedding_rows,
            covered_aspects=
                covered_aspects,
            total_product_aspects=
                total_product_aspects,
            covered_priority_groups=
                covered_priority_groups,
            priority_groups=
                priority_groups,
        )

        return {
            "idx": idx,
            "tie_usefulness": float(
                row[
                    "usefulness_norm"
                ]
            ),
            "tie_representativeness": float(
                row[
                    "representativeness_norm"
                ]
            ),
            **diagnostics,
        }

    reserve_limit = min(
        reserved_slots,
        budget,
    )

    while (
        len(selected_indices)
        < reserve_limit
    ):
        uncovered_priority = (
            priority_groups
            - covered_priority_groups
        )

        if not uncovered_priority:
            break

        eligible = []

        for idx, row in (
            product_df.iterrows()
        ):
            if idx in selected_indices:
                continue

            candidate_groups = parse_pipe_set(
                row[
                    "supported_minority_groups"
                ]
            )

            if (
                candidate_groups
                & uncovered_priority
            ):
                eligible.append(
                    evaluate_candidate(
                        idx
                    )
                )

        if not eligible:
            break

        best = max(
            eligible,
            key=lambda item: (
                item["minority_gain"],
                item["selection_score"],
                item["coverage_gain"],
                item["tie_usefulness"],
                item["tie_representativeness"],
            ),
        )

        best_idx = best["idx"]
        best_row = product_df.loc[
            best_idx
        ]

        selected_indices.append(
            best_idx
        )

        selected_embedding_rows.append(
            int(
                best_row[
                    "embedding_row"
                ]
            )
        )

        covered_aspects |= parse_pipe_set(
            best_row[
                "aspect_categories"
            ]
        )

        covered_priority_groups |= (
            parse_pipe_set(
                best_row[
                    "supported_minority_groups"
                ]
            )
            & priority_groups
        )

    target_count = min(
        budget,
        len(product_df),
    )

    while (
        len(selected_indices)
        < target_count
    ):
        candidates = []

        for idx, row in (
            product_df.iterrows()
        ):
            if idx in selected_indices:
                continue

            candidates.append(
                evaluate_candidate(idx)
            )

        best = max(
            candidates,
            key=lambda item: (
                item["selection_score"],
                item["minority_gain"],
                item["coverage_gain"],
                item["tie_usefulness"],
                item["tie_representativeness"],
            ),
        )

        best_idx = best["idx"]
        best_row = product_df.loc[
            best_idx
        ]

        selected_indices.append(
            best_idx
        )

        selected_embedding_rows.append(
            int(
                best_row[
                    "embedding_row"
                ]
            )
        )

        covered_aspects |= parse_pipe_set(
            best_row[
                "aspect_categories"
            ]
        )

        covered_priority_groups |= (
            parse_pipe_set(
                best_row[
                    "supported_minority_groups"
                ]
            )
            & priority_groups
        )

    return (
        product_df.loc[
            selected_indices
        ]
        .copy()
        .reset_index(drop=True)
    )


In [20]:
def summarize_reserved_slot_selection(
    selection_df,
    reserved_slots,
):
    records = []

    for product_id, full_group in (
        selector_ready_df
        .groupby("parent_asin")
    ):
        selected_group = selection_df[
            selection_df[
                "parent_asin"
            ] == product_id
        ]

        all_aspects = set()
        for value in full_group[
            "aspect_categories"
        ]:
            all_aspects |= parse_pipe_set(
                value
            )

        selected_aspects = set()
        for value in selected_group[
            "aspect_categories"
        ]:
            selected_aspects |= parse_pipe_set(
                value
            )

        all_supported = (
            supported_keys_by_product
            .get(
                product_id,
                set(),
            )
        )

        selected_supported = set()
        for value in selected_group[
            "supported_minority_groups"
        ]:
            selected_supported |= parse_pipe_set(
                value
            )

        priority_groups = (
            priority_minority_by_product
            .get(
                product_id,
                set(),
            )
        )

        records.append(
            {
                "reserved_slots":
                    reserved_slots,
                "parent_asin":
                    product_id,
                "mean_usefulness_norm":
                    float(
                        selected_group[
                            "usefulness_norm"
                        ].mean()
                    ),
                "mean_representativeness_norm":
                    float(
                        selected_group[
                            "representativeness_norm"
                        ].mean()
                    ),
                "aspect_coverage":
                    float(
                        len(
                            selected_aspects
                        )
                        /
                        max(
                            len(all_aspects),
                            1,
                        )
                    ),
                "supported_minority_recall":
                    float(
                        len(
                            selected_supported
                            & all_supported
                        )
                        /
                        len(all_supported)
                    )
                    if all_supported
                    else np.nan,
                "priority_minority_coverage":
                    float(
                        len(
                            selected_supported
                            & priority_groups
                        )
                        /
                        len(priority_groups)
                    )
                    if priority_groups
                    else np.nan,
                "mean_pairwise_similarity":
                    float(
                        mean_pairwise_similarity(
                            selected_group[
                                "embedding_row"
                            ].tolist()
                        )
                    ),
                "opinion_js_divergence":
                    float(
                        product_opinion_jsd(
                            product_id,
                            selected_group[
                                "review_id"
                            ].tolist(),
                        )
                    ),
            }
        )

    return pd.DataFrame(
        records
    )


In [21]:
reserved_slot_product_frames = []
reserved_slot_selection_frames = []

for reserved_slots in (
    RESERVED_SLOT_VALUES
):
    product_selections = []

    for product_id, group in (
        selector_ready_df
        .groupby("parent_asin")
    ):
        selected = (
            select_with_reserved_slots(
                product_df=group,
                budget=SELECT_K,
                reserved_slots=
                    reserved_slots,
            )
        )

        selected[
            "reserved_slots"
        ] = reserved_slots

        product_selections.append(
            selected
        )

    selection_df = pd.concat(
        product_selections,
        ignore_index=True,
    )

    reserved_slot_selection_frames.append(
        selection_df
    )

    reserved_slot_product_frames.append(
        summarize_reserved_slot_selection(
            selection_df,
            reserved_slots,
        )
    )

reserved_slot_selected_reviews_df = (
    pd.concat(
        reserved_slot_selection_frames,
        ignore_index=True,
    )
)

reserved_slot_product_df = (
    pd.concat(
        reserved_slot_product_frames,
        ignore_index=True,
    )
)

reserved_slot_overall_df = (
    reserved_slot_product_df
    .groupby(
        "reserved_slots",
        as_index=False,
    )
    .agg(
        mean_usefulness_norm=(
            "mean_usefulness_norm",
            "mean",
        ),
        mean_representativeness_norm=(
            "mean_representativeness_norm",
            "mean",
        ),
        mean_aspect_coverage=(
            "aspect_coverage",
            "mean",
        ),
        mean_supported_minority_recall=(
            "supported_minority_recall",
            "mean",
        ),
        mean_priority_minority_coverage=(
            "priority_minority_coverage",
            "mean",
        ),
        mean_pairwise_similarity=(
            "mean_pairwise_similarity",
            "mean",
        ),
        mean_opinion_js_divergence=(
            "opinion_js_divergence",
            "mean",
        ),
    )
)

display(
    reserved_slot_overall_df
)


,reserved_slots,mean_usefulness_norm,mean_representativeness_norm,mean_aspect_coverage,mean_supported_minority_recall,mean_priority_minority_coverage,mean_pairwise_similarity,mean_opinion_js_divergence
0,0,0.766217,0.776508,0.983333,0.379320,0.85,0.494914,0.322201
1,1,0.773176,0.782377,0.983333,0.379320,0.80,0.503748,0.318457
2,2,0.742725,0.766597,0.933333,0.416744,1.00,0.492611,0.336771
3,3,0.742725,0.766597,0.933333,0.416744,1.00,0.492611,0.336771


### Reserved-Slot Statistical Check

The current prototype uses 2 reserved slots.

The following exploratory paired tests compare the current setting (`2`) against no explicit reservation (`0`) for:

- usefulness,
- supported-minority recall,
- priority-minority coverage.

This isolates the reservation mechanism while keeping the minority score weight unchanged.


In [22]:
def reserved_slot_paired_test(
    slot_a,
    slot_b,
    metric,
    direction,
    seed,
):
    a = (
        reserved_slot_product_df[
            reserved_slot_product_df[
                "reserved_slots"
            ] == slot_a
        ][
            ["parent_asin", metric]
        ]
        .rename(
            columns={
                metric: "a"
            }
        )
    )

    b = (
        reserved_slot_product_df[
            reserved_slot_product_df[
                "reserved_slots"
            ] == slot_b
        ][
            ["parent_asin", metric]
        ]
        .rename(
            columns={
                metric: "b"
            }
        )
    )

    paired = (
        a.merge(
            b,
            on="parent_asin",
            validate="one_to_one",
        )
        .dropna()
    )

    statistic, p_value = safe_wilcoxon(
        paired["a"],
        paired["b"],
    )

    (
        mean_benefit,
        ci_low,
        ci_high,
    ) = paired_bootstrap_ci(
        paired["a"],
        paired["b"],
        direction=direction,
        n_boot=10000,
        seed=seed,
    )

    return {
        "slot_a": slot_a,
        "slot_b": slot_b,
        "metric": metric,
        "mean_a": float(
            paired["a"].mean()
        ),
        "mean_b": float(
            paired["b"].mean()
        ),
        "mean_benefit_difference":
            float(mean_benefit),
        "benefit_ci_95_low":
            float(ci_low),
        "benefit_ci_95_high":
            float(ci_high),
        "wilcoxon_statistic":
            float(statistic),
        "wilcoxon_p":
            float(p_value),
    }


reserved_slot_tests_df = pd.DataFrame(
    [
        reserved_slot_paired_test(
            2,
            0,
            "mean_usefulness_norm",
            1,
            1001,
        ),
        reserved_slot_paired_test(
            2,
            0,
            "supported_minority_recall",
            1,
            1002,
        ),
        reserved_slot_paired_test(
            2,
            0,
            "priority_minority_coverage",
            1,
            1003,
        ),
    ]
)

reserved_slot_tests_df[
    "holm_p"
] = holm_adjust(
    reserved_slot_tests_df[
        "wilcoxon_p"
    ]
)

display(
    reserved_slot_tests_df
)


,slot_a,slot_b,metric,mean_a,mean_b,mean_benefit_difference,benefit_ci_95_low,benefit_ci_95_high,wilcoxon_statistic,wilcoxon_p,holm_p
0,2,0,mean_usefulness_norm,0.742725,0.766217,-0.023493,-0.054056,0.003752,1.0,0.25,0.75
1,2,0,supported_minority_recall,0.416744,0.379320,0.037424,-0.008333,0.091515,1.0,0.25,0.75
2,2,0,priority_minority_coverage,1.000000,0.850000,0.150000,0.000000,0.300000,0.0,0.25,0.75


## 6. PP1-Ready Evidence Tables

These compact tables are intended for progress-presentation evidence.

They should be described as **prototype evidence** and not as final validated research findings.


In [23]:
KEY_MAIN_PAIRS = [
    (
        "PredictedUsefulnessTopK",
        "mean_usefulness_norm",
    ),
    (
        "PredictedUsefulnessTopK",
        "supported_minority_recall",
    ),
    (
        "PredictedUsefulnessTopK",
        "priority_minority_coverage",
    ),
    (
        "ProposedNoMinority",
        "supported_minority_recall",
    ),
    (
        "ProposedNoMinority",
        "priority_minority_coverage",
    ),
    (
        "HelpfulnessMMR",
        "mean_pairwise_similarity",
    ),
]

pp1_main_evidence_df = (
    main_stat_tests_df[
        main_stat_tests_df.apply(
            lambda row: (
                row[
                    "selector_b"
                ],
                row[
                    "metric"
                ],
            )
            in KEY_MAIN_PAIRS,
            axis=1,
        )
    ][
        [
            "selector_b",
            "metric_label",
            "mean_a",
            "mean_b",
            "mean_benefit_difference",
            "benefit_ci_95_low",
            "benefit_ci_95_high",
            "wilcoxon_p",
            "holm_p",
        ]
    ]
    .reset_index(drop=True)
)

display(
    pp1_main_evidence_df
)


,selector_b,metric_label,mean_a,mean_b,mean_benefit_difference,benefit_ci_95_low,benefit_ci_95_high,wilcoxon_p,holm_p
0,PredictedUsefulnessTopK,Predicted usefulness,0.742725,0.859078,-0.116353,-0.152127,-0.079277,0.001953,0.078125
1,PredictedUsefulnessTopK,Supported-minority recall,0.416744,0.278294,0.138450,0.074475,0.208803,0.007812,0.203125
2,PredictedUsefulnessTopK,Priority-minority coverage,1.000000,0.100000,0.900000,0.750000,1.000000,0.001953,0.078125
3,HelpfulnessMMR,Pairwise similarity,0.492611,0.438294,-0.054316,-0.109153,-0.001513,0.160156,1.000000
4,ProposedNoMinority,Supported-minority recall,0.416744,0.281072,0.135672,0.082051,0.195470,0.003906,0.117188
5,ProposedNoMinority,Priority-minority coverage,1.000000,0.200000,0.800000,0.650000,0.950000,0.001953,0.078125


In [24]:
pp1_ablation_evidence_df = (
    ablation_stat_tests_df[
        [
            "selector_b",
            "metric_label",
            "mean_a",
            "mean_b",
            "mean_benefit_difference",
            "benefit_ci_95_low",
            "benefit_ci_95_high",
            "wilcoxon_p",
            "holm_p",
        ]
    ]
    .copy()
)

display(
    pp1_ablation_evidence_df
)


,selector_b,metric_label,mean_a,mean_b,mean_benefit_difference,benefit_ci_95_low,benefit_ci_95_high,wilcoxon_p,holm_p
0,NoMinority,Supported-minority recall,0.416744,0.281072,0.135672,0.082168,0.195125,0.003906,0.011719
1,NoMinority,Priority-minority coverage,1.000000,0.200000,0.800000,0.650000,0.950000,0.001953,0.009766
2,NoCoverage,Aspect coverage,0.933333,0.833333,0.100000,0.016667,0.183333,0.125000,0.218750
3,NoRedundancy,Pairwise similarity,0.492611,0.514537,0.021926,0.000693,0.051679,0.109375,0.218750
4,NoRepresentativeness,Semantic representativeness,0.766597,0.647577,0.119019,0.081538,0.154691,0.001953,0.009766


## 7. Save Statistical Validation Outputs


In [25]:
MAIN_STATS_PATH = (
    STAT_DIR
    / "main_baseline_statistical_tests.csv"
)

ABLATION_STATS_PATH = (
    STAT_DIR
    / "ablation_statistical_tests.csv"
)

RESERVED_SLOT_PRODUCT_PATH = (
    STAT_DIR
    / "reserved_slot_sensitivity_by_product.csv"
)

RESERVED_SLOT_OVERALL_PATH = (
    STAT_DIR
    / "reserved_slot_sensitivity_overall.csv"
)

RESERVED_SLOT_SELECTIONS_PATH = (
    STAT_DIR
    / "reserved_slot_selected_reviews.csv"
)

RESERVED_SLOT_TESTS_PATH = (
    STAT_DIR
    / "reserved_slot_statistical_tests.csv"
)

PP1_MAIN_EVIDENCE_PATH = (
    STAT_DIR
    / "pp1_main_evidence.csv"
)

PP1_ABLATION_EVIDENCE_PATH = (
    STAT_DIR
    / "pp1_ablation_evidence.csv"
)

CONFIG_PATH = (
    STAT_DIR
    / "statistical_validation_config.json"
)


In [26]:
main_stat_tests_df.to_csv(
    MAIN_STATS_PATH,
    index=False,
)

ablation_stat_tests_df.to_csv(
    ABLATION_STATS_PATH,
    index=False,
)

reserved_slot_product_df.to_csv(
    RESERVED_SLOT_PRODUCT_PATH,
    index=False,
)

reserved_slot_overall_df.to_csv(
    RESERVED_SLOT_OVERALL_PATH,
    index=False,
)

reserved_slot_selected_reviews_df.to_csv(
    RESERVED_SLOT_SELECTIONS_PATH,
    index=False,
)

reserved_slot_tests_df.to_csv(
    RESERVED_SLOT_TESTS_PATH,
    index=False,
)

pp1_main_evidence_df.to_csv(
    PP1_MAIN_EVIDENCE_PATH,
    index=False,
)

pp1_ablation_evidence_df.to_csv(
    PP1_ABLATION_EVIDENCE_PATH,
    index=False,
)

stat_config = {
    "products": 10,
    "bootstrap_iterations": 10000,
    "bootstrap_ci": 0.95,
    "main_multiple_testing_correction":
        "Holm across all main baseline tests",
    "ablation_multiple_testing_correction":
        "Holm across targeted ablation tests",
    "reserved_slot_values":
        RESERVED_SLOT_VALUES,
    "current_reserved_slots": 2,
    "selection_budget_k": SELECT_K,
    "full_weights": FULL_WEIGHTS,
    "interpretation_warning":
        (
            "C1-derived aspect/sentiment, minority, "
            "coverage and fidelity statistics use synthetic "
            "integration-test labels and are prototype "
            "diagnostics only."
        ),
    "small_sample_warning":
        (
            "Only 10 products are available in the current "
            "smartphone pilot; statistical tests are exploratory."
        ),
}

with open(
    CONFIG_PATH,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        stat_config,
        f,
        indent=2,
    )

for path in [
    MAIN_STATS_PATH,
    ABLATION_STATS_PATH,
    RESERVED_SLOT_PRODUCT_PATH,
    RESERVED_SLOT_OVERALL_PATH,
    RESERVED_SLOT_SELECTIONS_PATH,
    RESERVED_SLOT_TESTS_PATH,
    PP1_MAIN_EVIDENCE_PATH,
    PP1_ABLATION_EVIDENCE_PATH,
    CONFIG_PATH,
]:
    print(
        "Saved:",
        path
    )


Saved: d:\Research\Repository\J26-DS-349\component_2\results\statistical_validation\main_baseline_statistical_tests.csv
Saved: d:\Research\Repository\J26-DS-349\component_2\results\statistical_validation\ablation_statistical_tests.csv
Saved: d:\Research\Repository\J26-DS-349\component_2\results\statistical_validation\reserved_slot_sensitivity_by_product.csv
Saved: d:\Research\Repository\J26-DS-349\component_2\results\statistical_validation\reserved_slot_sensitivity_overall.csv
Saved: d:\Research\Repository\J26-DS-349\component_2\results\statistical_validation\reserved_slot_selected_reviews.csv
Saved: d:\Research\Repository\J26-DS-349\component_2\results\statistical_validation\reserved_slot_statistical_tests.csv
Saved: d:\Research\Repository\J26-DS-349\component_2\results\statistical_validation\pp1_main_evidence.csv
Saved: d:\Research\Repository\J26-DS-349\component_2\results\statistical_validation\pp1_ablation_evidence.csv
Saved: d:\Research\Repository\J26-DS-349\component_2\results\st

## 8. Final Validation


In [27]:
assert len(
    main_stat_tests_df
) == (
    len(MAIN_COMPARATORS)
    * len(MAIN_METRICS)
)

assert len(
    ablation_stat_tests_df
) == len(
    ABLATION_TARGETS
)

assert set(
    reserved_slot_overall_df[
        "reserved_slots"
    ]
) == set(
    RESERVED_SLOT_VALUES
)

assert (
    reserved_slot_product_df[
        "parent_asin"
    ].nunique()
    == 10
)

assert (
    reserved_slot_selected_reviews_df
    .groupby(
        [
            "reserved_slots",
            "parent_asin",
        ]
    )
    .size()
    .eq(
        SELECT_K
    )
    .all()
)

assert (
    main_stat_tests_df[
        "n_products"
    ]
    .eq(10)
    .all()
)

assert (
    ablation_stat_tests_df[
        "n_products"
    ]
    .eq(10)
    .all()
)

for path in [
    MAIN_STATS_PATH,
    ABLATION_STATS_PATH,
    RESERVED_SLOT_PRODUCT_PATH,
    RESERVED_SLOT_OVERALL_PATH,
    RESERVED_SLOT_SELECTIONS_PATH,
    RESERVED_SLOT_TESTS_PATH,
    PP1_MAIN_EVIDENCE_PATH,
    PP1_ABLATION_EVIDENCE_PATH,
    CONFIG_PATH,
]:
    assert path.exists()

print(
    "Statistical validation and PP1 evidence generation passed."
)


Statistical validation and PP1 evidence generation passed.


## 9. Milestone Summary

### Completed

- Paired product-level Wilcoxon tests
- Paired bootstrap 95% confidence intervals
- Holm multiple-comparison correction
- Full Proposed vs five baseline methods
- Targeted ablation validation
- Minority reserved-slot sensitivity
- PP1-ready evidence tables
- Reproducible statistical output artifacts

### Interpretation Rule

The current notebook provides **prototype statistical evidence**, not final research validation.

Final research conclusions must wait for:

- real C1 outputs,
- broader data coverage,
- final held-out evaluation,
- final threshold/weight selection,
- and, where appropriate, stronger statistical power.
